## Incorporating VL-SAE to the inference process of LVLMs

In [1]:
import torch
import os
import json
import sys
import os
os.environ['CUDA_VISIBLE_DEVICES'] = "3"
sys.path.append("../sae_trainer")
sys.path.append("../VCD")
sys.path.append("../VCD/experiments")
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
from llava.conversation import conv_templates, SeparatorStyle
from llava.model.builder import load_pretrained_model
from llava.utils import disable_torch_init
from llava.mm_utils import tokenizer_image_token, get_model_name_from_path, KeywordsStoppingCriteria

from sae_model import VL_SAE, AuxiliaryAE
from utils import display_concepts

from PIL import Image
import matplotlib.pyplot as plt
from functools import partial

from vcd_utils.vlsae_sample import evolve_vlsae_sampling
evolve_vlsae_sampling()

In [ ]:
class FeatureExtractor:
    def __init__(self, model, layer_names, sae_path=None, alignment_path=None, input_dim=4096, topk=128, hidden_ratio=8, modify_fn=None):
        self.model = model
        self.features = {}
        self.layer_names = layer_names if isinstance(layer_names, (list, tuple)) else [layer_names]
        self.hooks = {}
        self.modify_fn = modify_fn
        self.input_dim = input_dim
        self.vis_indices = None
        
        if sae_path:
            self.sae = VL_SAE(input_dim, hidden_dim=hidden_ratio*input_dim, topk=topk).cuda().half()
            self.alignment_model = AuxiliaryAE(input_dim, input_dim, projection_dim=4096).cuda().half()
            self.sae.load_state_dict(torch.load(sae_path))
            print(f'Successfully loaded SAE model from {sae_path}')
            self.alignment_model.load_state_dict(torch.load(alignment_path))
            print(f'Successfully loaded Auxiliary AE model from {alignment_path}')
            self.sae.eval()
            self.alignment_model.eval()

    def set_model(self, model):
        self.model = model

    def hook_fn(self, name):
        def hook(module, input, output):
            if self.vis_indices is not None:
                hidden_states = output[0]
                
                vis_features = hidden_states[:, self.vis_indices, :]
                text_features = hidden_states[:, self.vis_indices[-1]+1:, :]
                vis_features_mean = vis_features.mean(dim=1) 
                text_features_mean = text_features.mean(dim=1)
                
                if self.modify_fn is not None:
                    modified_vis_features_mean, modified_text_features_mean = self.modify_fn(
                        vis_features_mean, text_features_mean, self.sae, self.alignment_model
                    )
                    hidden_states[:, self.vis_indices, :] = hidden_states[:, self.vis_indices, :] + \
                        modified_vis_features_mean.unsqueeze(1) - vis_features_mean.unsqueeze(1)
                    hidden_states[:, self.vis_indices[-1]+1:, :] = hidden_states[:, self.vis_indices[-1]+1:, :] + \
                        modified_text_features_mean.unsqueeze(1) - text_features_mean.unsqueeze(1)
                    return (hidden_states,)
            return output
        return hook
    
    def _get_layer(self, name):
        if '.' in name:
            module = self.model
            for part in name.split('.')[:-1]:
                module = getattr(module, part)
            return getattr(module, name.split('.')[-1])
        return getattr(self.model, name)

    def add_hooks(self, layer_names=None):
        if layer_names is None:
            layer_names = self.layer_names
        elif isinstance(layer_names, str):
            layer_names = [layer_names]
            
        for name in layer_names:
            if name not in self.hooks:
                layer = self._get_layer(name)
                self.hooks[name] = layer.register_forward_hook(self.hook_fn(name))
                # print(f"Added hook to layer: {name}")
    
    def remove_hooks(self, layer_names=None):
        if layer_names is None:
            layer_names = list(self.hooks.keys())
        elif isinstance(layer_names, str):
            layer_names = [layer_names]
            
        for name in layer_names:
            if name in self.hooks:
                self.hooks[name].remove()
                del self.hooks[name]
                # print(f"Removed hook from layer: {name}")
    
    def set_vis_indices(self, indices):
        self.vis_indices = indices

In [ ]:
# modify function: how to modify the representations of LVLMs using sae
def directly_sae_forward(vision_features, text_features, sae):
    vision_features, text_features = sae(vision_features, text_features)
    return vision_features, text_features
def weight_sae_forward(vision_features, text_features, sae, alpha=0.1):
    recon_vision_features, recon_text_features = sae(vision_features, text_features)
    return vision_features + alpha * recon_vision_features, text_features + alpha * recon_text_features
def image_constrain_text(vision_features, text_features, sae, alignment_model, alpha=1, beta=0.5):
    vision_embeds, text_embeds = alignment_model.encoder(vision_features, text_features)
    vision_embeds, text_embeds = sae.encode(vision_embeds), sae.encode(text_embeds)
    text_embeds = (1-beta)*text_embeds+beta*vision_embeds
    recon_text_features = alignment_model.decoder(text_embed=sae.text_decoder(text_embeds))[-1]
    text_features = (1-alpha)*text_features + alpha*recon_text_features
    return vision_features, text_features

In [ ]:
model_path = "../pretrained_models/llava-v1.5-7b"
extract_layers = 'model.layers.30'
sae_path = '../sae_trainer/sae_weights/llava_256_8_best.pth'
alignment_path = '../sae_trainer/sae_weights/llava_aux_best.pt'
modify_alpha = 0.8
modify_beta = 0.9

disable_torch_init()
model_path = os.path.expanduser(model_path)
model_name = get_model_name_from_path(model_path)
tokenizer, model, image_processor, context_len = load_pretrained_model(model_path, None, model_name)

# modify the LVLM representations using image_constrain_text
modify_fn = partial(image_constrain_text, alpha=modify_alpha, beta=modify_beta)

feature_extractor = FeatureExtractor(None, extract_layers, sae_path=sae_path, alignment_path=alignment_path, topk=256, hidden_ratio=8, modify_fn=modify_fn)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Successfully loaded SAE model from ../sae_trainer/sae_weights/llava_256_8_best.pth
Successfully loaded Auxiliary AE model from ../sae_trainer/sae_weights/llava_aux_best.pt


In [4]:
# Text Input
qs = "Please describe this image in detail."
if model.config.mm_use_im_start_end:
    qs = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN + '\n' + qs
else:
    qs = DEFAULT_IMAGE_TOKEN + '\n' + qs
conv = conv_templates['llava_v1'].copy()
conv.append_message(conv.roles[0], qs)
conv.append_message(conv.roles[1], None)
prompt = conv.get_prompt()
input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda()
stop_str = conv.sep if conv.sep_style != SeparatorStyle.TWO else conv.sep2
keywords = [stop_str]
stopping_criteria = KeywordsStoppingCriteria(keywords, tokenizer, input_ids)

# Image Input
image_path = "./demo_input.jpg"
image = Image.open(image_path)
image_tensor = image_processor.preprocess(image, return_tensors='pt')['pixel_values'][0]

In [5]:
image_start_idx = torch.where(input_ids[0] == IMAGE_TOKEN_INDEX)[0].item()
image_tokens_seq_len = 576
vis_token_indices = list(range(image_start_idx, image_start_idx+image_tokens_seq_len))
    
with torch.inference_mode():
    output_ids = model.generate(
        input_ids,
        images=image_tensor.unsqueeze(0).half().cuda(),
        cd_alpha=0.6,
        cd_beta=0.8,
        do_sample=True,
        temperature=1.0,
        use_sae=True,
        feature_extractor=feature_extractor,
        vis_indices=vis_token_indices,
        top_p=1,
        top_k=None,
        max_new_tokens=1024,
        use_cache=False)

In [6]:
input_token_len = input_ids.shape[1]
n_diff_input_output = (input_ids != output_ids[:, :input_token_len]).sum().item()
if n_diff_input_output > 0:
    print(f'[Warning] {n_diff_input_output} output_ids are not the same as the input_ids')
outputs = tokenizer.batch_decode(output_ids[:, input_token_len:], skip_special_tokens=True)[0]
outputs = outputs.strip()
if outputs.endswith(stop_str):
    outputs = outputs[:-len(stop_str)]
outputs = outputs.strip()
print(outputs)

The image features a kitchen with a stainless steel oven prominently placed in the center of the scene. The oven is open, revealing its interior. The kitchen counter is cluttered with various items, including a wine glass, a cup, and a spoon. There are also two bottles placed on the counter, one near the left side and the other closer to the center.

In addition to the oven, there is a microwave positioned above the oven, and a sink can be seen on the left side of the kitchen. A towel is hanging on the oven door, and a bowl is placed on the counter near the center. The overall atmosphere of the kitchen suggests that it is in use and has been recently prepared for cooking or baking.
